# Couche Silver KBO Open Data : `entreprise` → `entreprise_silver`

## 0. Contexte : bronze vs silver

La couche **bronze** (`entreprise`) reste sous forme **brute** :

- des codes non traduits (`Status="AC"`, `TypeOfAddress="REGO"`...) ;
- des tableaux indexés `0/1/2/3` sans clé porteuse de sens ;
- des champs dupliqués par langue (`CountryNL`/`CountryFR`, `MunicipalityNL`/`MunicipalityFR`...) ;
- des activités qui réapparaissent en double sous plusieurs versions NACE (2003, 2008, 2025) pour la même réalité.

La couche **silver** de transformation géres tout ça

## 1. Lecture du bronze

Charger `entreprise` et `kbo_code` filtré sur `Language="FR"`.

In [7]:
!pip install pyspark

You should consider upgrading via the '/Users/theo-dev/Dev/M2_IPSSI/M2_BIGDATA/.venv/bin/python3 -m pip install --upgrade pip' command.


In [8]:
from pyspark.sql.functions import col, coalesce, lit

# Initialisation de la session Spark
spark = SparkSession.builder \
    .appName("KBO_Silver_Enterprise") \
    .getOrCreate()

# Chemins d'accès basés sur votre arborescence
import csv

# Initialisation du dictionnaire global
translations = {}

# Remplace le chemin par le bon si nécessaire
path_code = "data/code.csv"

# Chargement du fichier code.csv en Python pur
with open(path_code, mode='r', encoding='utf-8') as file:
    reader = csv.DictReader(file)
    for row in reader:
        # On ne garde que les traductions françaises
        if row.get("Language") == "FR":
            category = row.get("Category")
            code = row.get("Code")
            description = row.get("Description")
            
            # Construction du dictionnaire imbriqué
            if category not in translations:
                translations[category] = {}
            translations[category][code] = description

print("Dictionnaire de traduction chargé avec succès !")


path_enterprise = "data/enterprise.csv"

# 1. Lecture du bronze : code.csv filtré sur "FR"



def translate(category_name, code_value):
    """
    Cherche la traduction FR d'un code pour une catégorie donnée.
    Retourne la traduction si elle existe, sinon retourne le code brut.
    """
    if not code_value:
        return ""
    
    # On navigue dans le dictionnaire en toute sécurité avec .get()
    return translations.get(category_name, {}).get(code_value, code_value)

NameError: name 'SparkSession' is not defined

## 2. Champs scalaires codés

`Status`, `JuridicalSituation`, `TypeOfEnterprise`, `JuridicalForm`, `JuridicalFormCAC` : cinq champs plats, chacun traduit indépendamment via `kbo_code`. Un champ absent ou vide dans le bronze (ex. `JuridicalFormCAC=""`) doit être **omis** du silver plutôt que d'y apparaître traduit en valeur nulle.

In [ ]:

{
  "_id": "0200.245.711",
  "enterpriseNumber": "0200.245.711",
  "startDate": "01-01-1922",
  "juridicalForm": "Société coopérative de droit public (ancien statut)",
  "juridicalSituation": "Dissolution volontaire – liquidation",
  "status": "Actif",
  "typeOfEnterprise": "Personne morale"
}

def process_scalars(bronze_doc):
    """
    Initialise le document silver avec les identifiants et les champs scalaires traduits.
    """
    silver_doc = {}
    
    # Identifiants de base
    silver_doc["_id"] = bronze_doc.get("_id")
    # Si EnterpriseNumber est absent, on utilise _id en secours comme spécifié dans ton schéma cible
    silver_doc["enterpriseNumber"] = bronze_doc.get("EnterpriseNumber", silver_doc["_id"])
    
    if "StartDate" in bronze_doc:
        silver_doc["startDate"] = bronze_doc.get("StartDate")
        
    # Liste des champs scalaires à traduire et leur nom final dans la couche silver
    scalar_fields_to_map = [
        ("Status", "status"),
        ("JuridicalSituation", "juridicalSituation"),
        ("TypeOfEnterprise", "typeOfEnterprise"),
        ("JuridicalForm", "juridicalForm"),
        ("JuridicalFormCAC", "juridicalFormCAC")
    ]
    
    for bronze_key, silver_key in scalar_fields_to_map:
        val = bronze_doc.get(bronze_key)
        # On ne crée la clé dans le silver QUE si la valeur existe dans le bronze
        if val: 
            silver_doc[silver_key] = translate(bronze_key, val)
            
    return silver_doc

## 3. Dénominations : tableau → dict `{type traduit: {language, denomination}}`

Chaque entrée doit être keyée par son `TypeOfDenomination` **traduit**

In [ ]:
{
  "_id": "0200.245.711",
  "enterpriseNumber": "0200.245.711",
  "startDate": "01-01-1922",
  "denominations": {
    "Abréviation": {
      "language": "néerlandais",
      "denomination": "DENDEROORD"
    },
    "Dénomination": {
      "language": "néerlandais",
      "denomination": "Intercommunaal Sanatorium Denderoord"
    }
  },
  "juridicalForm": "Société coopérative de droit public (ancien statut)",
  "juridicalSituation": "Dissolution volontaire – liquidation",
  "status": "Actif",
  "typeOfEnterprise": "Personne morale"
}

{'_id': '0200.245.711',
 'enterpriseNumber': '0200.245.711',
 'startDate': '01-01-1922',
 'denominations': {'Abréviation': {'language': 'néerlandais',
   'denomination': 'DENDEROORD'},
  'Dénomination': {'language': 'néerlandais',
   'denomination': 'Intercommunaal Sanatorium Denderoord'}},
 'juridicalForm': 'Société coopérative de droit public (ancien statut)',
 'juridicalSituation': 'Dissolution volontaire – liquidation',
 'status': 'Actif',
 'typeOfEnterprise': 'Personne morale'}

## 4. Adresses : tableau → dict `{type traduit: {country, street...}}`

Même principe clé-traduite que les dénominations, plus deux règles spécifiques :

- **`country`** : `CountryFR` nettoyé des mentions entre parenthèses (`"France (Métropole)"` → `"France"`) et des espaces multiples ; si le résultat est vide, mettre `"Belgique"` par défaut.
- **Champs vides omis** : `box`, `zipcode`... vides ne doivent pas apparaître dans le document silver.

In [ ]:
{
  "_id": "0200.245.711",
  "enterpriseNumber": "0200.245.711",
  "startDate": "01-01-1922",
  "denominations": {
    "Abréviation": {
      "language": "néerlandais",
      "denomination": "DENDEROORD"
    },
    "Dénomination": {
      "language": "néerlandais",
      "denomination": "Intercommunaal Sanatorium Denderoord"
    }
  },
  "addresses": {
    "Siège": {
      "country": "Belgique",
      "zipcode": "9500",
      "municipality": "Geraardsbergen",
      "street": "Hoge Buizemont",
      "houseNumber": "247",
      "box": ""
    }
  },
  "juridicalForm": "Société coopérative de droit public (ancien statut)",
  "juridicalSituation": "Dissolution volontaire – liquidation",
  "status": "Actif",
  "typeOfEnterprise": "Personne morale"
}

{'_id': '0200.245.711',
 'enterpriseNumber': '0200.245.711',
 'startDate': '01-01-1922',
 'denominations': {'Abréviation': {'language': 'néerlandais',
   'denomination': 'DENDEROORD'},
  'Dénomination': {'language': 'néerlandais',
   'denomination': 'Intercommunaal Sanatorium Denderoord'}},
 'addresses': {'Siège': {'country': 'Belgique',
   'zipcode': '9500',
   'municipality': 'Geraardsbergen',
   'street': 'Hoge Buizemont',
   'houseNumber': '247',
   'box': ''}},
 'juridicalForm': 'Société coopérative de droit public (ancien statut)',
 'juridicalSituation': 'Dissolution volontaire – liquidation',
 'status': 'Actif',
 'typeOfEnterprise': 'Personne morale'}

## 5. Contacts : tableau → dict `{email, phone, web}`

`EntityContact` (indique juste si le contact appartient à l'entreprise, un établissement ou une succursale) ne doit **jamais** être repris dans le silver. Un contact avec une valeur vide doit être ignoré.

In [ ]:
{
  "_id": "0201.543.234",
  "enterpriseNumber": "0201.543.234",
  "startDate": "26-10-1948",
  "denominations": {
    "Dénomination": {
      "language": "français",
      "denomination": "TIBI"
    }
  },
  "addresses": {
    "Siège": {
      "country": "Belgique",
      "zipcode": "6010",
      "municipality": "Charleroi",
      "street": "Rue du Déversoir",
      "houseNumber": "1",
      "box": ""
    }
  },
  "contacts": {
    "phone": "071 44 00 40",
    "email": "officiel.ic-tibi@tibi.be",
    "fax": "071 36 04 84"
  },
  "juridicalForm": "Société coopérative de droit public",
  "juridicalSituation": "Situation normale",
  "status": "Actif",
  "typeOfEnterprise": "Personne morale"
}

{'_id': '0201.543.234',
 'enterpriseNumber': '0201.543.234',
 'startDate': '26-10-1948',
 'denominations': {'Dénomination': {'language': 'français',
   'denomination': 'TIBI'}},
 'addresses': {'Siège': {'country': 'Belgique',
   'zipcode': '6010',
   'municipality': 'Charleroi',
   'street': 'Rue du Déversoir',
   'houseNumber': '1',
   'box': ''}},
 'contacts': {'phone': '071 44 00 40',
  'email': 'officiel.ic-tibi@tibi.be',
  'fax': '071 36 04 84'},
 'juridicalForm': 'Société coopérative de droit public',
 'juridicalSituation': 'Situation normale',
 'status': 'Actif',
 'typeOfEnterprise': 'Personne morale'}

In [ ]:
import re

def process_denominations(denominations_bronze):
    """Transforme la liste de dénominations en dict indexé par le type traduit."""
    denoms_silver = {}
    for d in denominations_bronze:
        t_type = translate("TypeOfDenomination", d.get("TypeOfDenomination"))
        # En cas de type dupliqué, la dernière valeur écrasera naturellement la précédente[cite: 1]
        denoms_silver[t_type] = {
            "language": d.get("Language"),
            "denomination": d.get("Denomination")
        }
    return denoms_silver

def clean_country(country_fr):
    """Nettoie le nom du pays (retrait des parenthèses et espaces multiples)."""
    if not country_fr:
        return "Belgique" # Par défaut[cite: 1]
    
    cleaned = re.sub(r'\(.*?\)', '', country_fr) #[cite: 1]
    cleaned = re.sub(r'\s+', ' ', cleaned).strip() #[cite: 1]
    return cleaned if cleaned else "Belgique" #[cite: 1]

def process_addresses(addresses_bronze):
    """Transforme la liste d'adresses en dict et nettoie le pays."""
    addrs_silver = {}
    for a in addresses_bronze:
        t_type = translate("TypeOfAddress", a.get("TypeOfAddress"))
        addr = {}
        
        # Mapping et exclusion des champs vides[cite: 1]
        country = clean_country(a.get("CountryFR"))
        if country: addr["country"] = country
        
        for field in ["Zipcode", "Municipality", "Street", "HouseNumber", "Box"]:
            val = a.get(field)
            if val:
                addr[field[0].lower() + field[1:]] = val
                
        addrs_silver[t_type] = addr
    return addrs_silver

def process_contacts(contacts_bronze):
    """Transforme les contacts en ignorant 'EntityContact' et les valeurs vides."""
    contacts_silver = {}
    for c in contacts_bronze:
        c_type = c.get("ContactType", "").lower()
        # 'EntityContact' ne doit jamais être repris[cite: 1]
        if c_type == "entitycontact": 
            continue # L'erreur de syntaxe venait d'ici !
            
        val = c.get("Value")
        if val: # Un contact avec une valeur vide doit être ignoré[cite: 1]
            contacts_silver[c_type] = val
    return contacts_silver

## 6. Activités : dédoublonnage inter-versions NACE + répartition main/secondary

Le point le plus subtil de toute la couche silver. Une même activité réelle est souvent codée sous **plusieurs versions NACE** (2003, 2008, 2025) avec des libellés différents mais qui décrivent la même chose. Règle : dédoublonner sur `(activityGroup, description)` et, en cas de collision, **garder la version NACE la plus récente**. Le `NaceCode` brut ne doit **jamais** être gardé dans le silver, une fois `description` résolue via `Nace{version}`, le code numérique ne sert plus à un lecteur humain. Répartir le résultat en `{main: [...], secondary: [...]}` selon `Classification`.

In [ ]:
{
  "_id": "0201.311.226",
  "enterpriseNumber": "0201.311.226",
  "startDate": "01-01-1968",
  "denominations": {
    "Dénomination": {
      "language": "néerlandais",
      "denomination": "FLUVIUS"
    }
  },
  "addresses": {
    "Siège": {
      "country": "Belgique",
      "zipcode": "3500",
      "municipality": "Hasselt",
      "street": "Trichterheideweg",
      "houseNumber": "8",
      "box": ""
    }
  },
  "contacts": {},
  "activities": {
    "main": [
      {
        "activityGroup": "Activités TVA",
        "description": "Distribution d'électricité",
        "naceVersion": "2008"
      },
      {
        "activityGroup": "Activités TVA",
        "description": "Distribution d’électricité",
        "naceVersion": "2025"
      },
      {
        "activityGroup": "Activités ONSS",
        "description": "Distribution d'électricité",
        "naceVersion": "2008"
      },
      {
        "activityGroup": "Activités ONSS",
        "description": "Distribution d’électricité",
        "naceVersion": "2025"
      },
      {
        "activityGroup": "Activités TVA",
        "description": "Production d'électricité",
        "naceVersion": "2003"
      }
    ],
    "secondary": [
      {
        "activityGroup": "Activités TVA",
        "description": "Activités de télécommunications filaires, sans fil et satellitaires",
        "naceVersion": "2025"
      },
      {
        "activityGroup": "Activités TVA",
        "description": "Production d’électricité à partir de sources non renouvelables",
        "naceVersion": "2025"
      },
      {
        "activityGroup": "Activités TVA",
        "description": "Collecte et traitement des eaux usées",
        "naceVersion": "2025"
      },
      {
        "activityGroup": "Activités TVA",
        "description": "Distribution de combustibles gazeux par conduites",
        "naceVersion": "2025"
      },
      {
        "activityGroup": "Activités TVA",
        "description": "Production d'électricité",
        "naceVersion": "2008"
      },
      {
        "activityGroup": "Activités TVA",
        "description": "Télécommunications sans fil",
        "naceVersion": "2008"
      }
    ]
  },

  "branches": {},
  "juridicalForm": "Association chargée de mission (Région flamande)",
  "juridicalSituation": "Situation normale",
  "status": "Actif",
  "typeOfEnterprise": "Personne morale"
}

def process_activities(activities_bronze):
    """Dédoublonne sur (activityGroup, description) et garde la NACE la plus récente[cite: 1]."""
    dedup_dict = {}
    
    for act in activities_bronze:
        group = act.get("ActivityGroup")
        desc = act.get("Description")
        version = act.get("NaceVersion", "0") # Ex: "2003", "2008", "2025"[cite: 1]
        classification = act.get("Classification", "main").lower()
        
        key = (group, desc)
        
        # En cas de collision, garder la version NACE la plus récente[cite: 1]
        if key not in dedup_dict or int(version) > int(dedup_dict[key]["naceVersion"]):
            # Le NaceCode numérique n'est jamais gardé[cite: 1]
            dedup_dict[key] = {
                "activityGroup": group,
                "description": desc,
                "naceVersion": version,
                "_classification": classification # champ temporaire pour le tri
            }
            
    # Répartition main/secondary[cite: 1]
    activities_silver = {"main": [], "secondary": []}[cite: 1]
    for act in dedup_dict.values():
        cat = "main" if act.pop("_classification") == "main" else "secondary"
        activities_silver[cat].append(act)
        
    return activities_silver

## 7. Établissements : mêmes règles, en plus léger

Un établissement a ses propres dénominations/adresses/contacts/activités, à nettoyer avec **exactement les mêmes règles** que ci-dessus. Seule différence avec le document entreprise : pas d'`EnterpriseNumber` (déjà le document de cette entreprise, ce serait une redondance pure). Résultat keyé par `EstablishmentNumber`.

In [ ]:
{
  "_id": "0201.311.226",
  "enterpriseNumber": "0201.311.226",
  "startDate": "01-01-1968",
  "denominations": {
    "Dénomination": {
      "language": "néerlandais",
      "denomination": "FLUVIUS"
    }
  },
  "addresses": {
    "Siège": {
      "country": "Belgique",
      "zipcode": "3500",
      "municipality": "Hasselt",
      "street": "Trichterheideweg",
      "houseNumber": "8",
      "box": ""
    }
  },
  "contacts": {},
  "activities": {
    "main": [
      {
        "activityGroup": "Activités TVA",
        "description": "Distribution d'électricité",
        "naceVersion": "2008"
      },
      {
        "activityGroup": "Activités TVA",
        "description": "Distribution d’électricité",
        "naceVersion": "2025"
      },
      {
        "activityGroup": "Activités ONSS",
        "description": "Distribution d'électricité",
        "naceVersion": "2008"
      },
      {
        "activityGroup": "Activités ONSS",
        "description": "Distribution d’électricité",
        "naceVersion": "2025"
      },
      {
        "activityGroup": "Activités TVA",
        "description": "Production d'électricité",
        "naceVersion": "2003"
      }
    ],
    "secondary": [
      {
        "activityGroup": "Activités TVA",
        "description": "Activités de télécommunications filaires, sans fil et satellitaires",
        "naceVersion": "2025"
      },
      {
        "activityGroup": "Activités TVA",
        "description": "Production d’électricité à partir de sources non renouvelables",
        "naceVersion": "2025"
      },
      {
        "activityGroup": "Activités TVA",
        "description": "Collecte et traitement des eaux usées",
        "naceVersion": "2025"
      },
      {
        "activityGroup": "Activités TVA",
        "description": "Distribution de combustibles gazeux par conduites",
        "naceVersion": "2025"
      },
      {
        "activityGroup": "Activités TVA",
        "description": "Production d'électricité",
        "naceVersion": "2008"
      },
      {
        "activityGroup": "Activités TVA",
        "description": "Télécommunications sans fil",
        "naceVersion": "2008"
      }
    ]
  },
  "establishments": {
    "2.158.307.210": {
      "startDate": "01-01-1968",
      "denominations": {
        "Dénomination commerciale": {
          "language": "néerlandais",
          "denomination": "FLUVIUS o.v."
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "3500",
          "municipality": "Hasselt",
          "street": "Trichterheideweg",
          "houseNumber": "8",
          "box": ""
        }
      },
      "contacts": {},
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSSAPL",
            "description": "Distribution et commerce d'électricité",
            "naceVersion": "2003"
          },
          {
            "activityGroup": "Activités ONSS",
            "description": "Commerce d’électricité",
            "naceVersion": "2025"
          },
          {
            "activityGroup": "Activités ONSS",
            "description": "Commerce d'électricité",
            "naceVersion": "2008"
          }
        ],
        "secondary": []
      }
    }
  },
  "branches": {},
  "juridicalForm": "Association chargée de mission (Région flamande)",
  "juridicalSituation": "Situation normale",
  "status": "Actif",
  "typeOfEnterprise": "Personne morale"
}

{'_id': '0201.311.226',
 'enterpriseNumber': '0201.311.226',
 'startDate': '01-01-1968',
 'denominations': {'Dénomination': {'language': 'néerlandais',
   'denomination': 'FLUVIUS'}},
 'addresses': {'Siège': {'country': 'Belgique',
   'zipcode': '3500',
   'municipality': 'Hasselt',
   'street': 'Trichterheideweg',
   'houseNumber': '8',
   'box': ''}},
 'contacts': {},
 'activities': {'main': [{'activityGroup': 'Activités TVA',
    'description': "Distribution d'électricité",
    'naceVersion': '2008'},
   {'activityGroup': 'Activités TVA',
    'description': 'Distribution d’électricité',
    'naceVersion': '2025'},
   {'activityGroup': 'Activités ONSS',
    'description': "Distribution d'électricité",
    'naceVersion': '2008'},
   {'activityGroup': 'Activités ONSS',
    'description': 'Distribution d’électricité',
    'naceVersion': '2025'},
   {'activityGroup': 'Activités TVA',
    'description': "Production d'électricité",
    'naceVersion': '2003'}],
  'secondary': [{'activityGro

## 8. Succursales : encore plus léger

Une succursale n'a jamais de dénomination propre ni d'activité propre : ne nettoyer que `addresses` et `contacts`. Ni `EnterpriseNumber`, ni `denominations`, ni `activities` dans la sortie. Résultat keyé par `Id`.

In [ ]:
{
  "_id": "0257.883.408",
  "enterpriseNumber": "0257.883.408",
  "startDate": "01-09-1995",
  "denominations": {
    "Dénomination": {
      "language": "français",
      "denomination": "ASSOCIATION TURQUE DES EXPORTATEURS DE TEXTILE ET D'HABILLEMENT D'ISTANBUL - ITKIB"
    }
  },
  "addresses": {
    "Siège": {
      "country": "Turquie",
      "zipcode": "34196",
      "municipality": "yenibosna - Istamboul",
      "street": "itkib bis ticaret komplexi b/blok coban cesme mekvil sanayi/caddesi",
      "houseNumber": "0",
      "box": ""
    }
  },
  "contacts": {},
  "activities": {
    "main": [],
    "secondary": []
  },
  "establishments": {
    "2.076.372.003": {
      "startDate": "02-05-1996",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "ITKIB"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "1040",
          "municipality": "Bruxelles",
          "street": "Rue de la Loi",
          "houseNumber": "28",
          "box": ""
        }
      },
      "contacts": {},
      "activities": {
        "main": [
          {
            "activityGroup": "Activités",
            "description": "Agences de presse",
            "naceVersion": "2003"
          },
          {
            "activityGroup": "Activités",
            "description": "Activités d’agence de presse",
            "naceVersion": "2025"
          }
        ],
        "secondary": []
      }
    }
  },
  "branches": {
    "9.000.006.626": {
      "startDate": "01-09-1995",
      "addresses": {
        "Succursale": {
          "country": "Belgique",
          "zipcode": "1040",
          "municipality": "Bruxelles",
          "street": "Rue de la Loi",
          "houseNumber": "28",
          "box": ""
        }
      },
      "contacts": {}
    }
  },
  "juridicalForm": "Entité étrangère",
  "juridicalSituation": "Situation normale",
  "status": "Actif",
  "typeOfEnterprise": "Personne morale"
}


def process_establishments(establishments_bronze):
    """Applique les mêmes règles que l'entreprise, sans l'EnterpriseNumber[cite: 1]."""
    ests_silver = {}
    for e in establishments_bronze:
        est_id = e.get("EstablishmentNumber")[cite: 1]
        est_data = {}
        
        if "StartDate" in e:
            est_data["startDate"] = e["StartDate"]
            
        est_data["denominations"] = process_denominations(e.get("Denominations", []))
        est_data["addresses"] = process_addresses(e.get("Addresses", []))
        est_data["contacts"] = process_contacts(e.get("Contacts", []))
        est_data["activities"] = process_activities(e.get("Activities", []))
        
        ests_silver[est_id] = est_data
    return ests_silver

def process_branches(branches_bronze):
    """Ne nettoie que addresses et contacts (jamais de dénomination ni d'activité)[cite: 1]."""
    branches_silver = {}
    for b in branches_bronze:
        branch_id = b.get("Id")[cite: 1]
        branch_data = {}
        
        if "StartDate" in b:
            branch_data["startDate"] = b["StartDate"]
            
        branch_data["addresses"] = process_addresses(b.get("Addresses", []))[cite: 1]
        branch_data["contacts"] = process_contacts(b.get("Contacts", []))[cite: 1]
        
        branches_silver[branch_id] = branch_data
    return branches_silver

## 9. Écriture dans `entreprise_silver`

Snapshot complet : vider `entreprise_silver` puis écrire le résultat.

In [ ]:
def transform_to_silver(bronze_doc):
    """Fonction principale de transformation d'un document entreprise brut en silver."""
    # 1. Base et scalaires
    silver_doc = process_scalars(bronze_doc)
    
    # 2. Dénominations, Adresses, Contacts, Activités
    silver_doc["denominations"] = process_denominations(bronze_doc.get("Denominations", []))
    silver_doc["addresses"] = process_addresses(bronze_doc.get("Addresses", []))
    silver_doc["contacts"] = process_contacts(bronze_doc.get("Contacts", []))
    silver_doc["activities"] = process_activities(bronze_doc.get("Activities", []))
    
    # 3. Établissements et succursales
    silver_doc["establishments"] = process_establishments(bronze_doc.get("Establishments", []))
    silver_doc["branches"] = process_branches(bronze_doc.get("Branches", []))
    
    return silver_doc

from pymongo import MongoClient

# Connexion explicite
client = MongoClient("mongodb://localhost:27017/")

# On pointe vers la BONNE base de données
db = client["kbo_db"]

# Vider la collection silver pour repartir au propre
db.entreprise_silver.delete_many({}) 

# On lance la transformation sur le Bronze
for doc in db.entreprise.find():
    silver_doc = transform_to_silver(doc)
    db.entreprise_silver.insert_one(silver_doc)

print(f"✅ Terminé ! {db.entreprise_silver.count_documents({})} documents insérés dans le silver.")

NameError: name 'cite' is not defined

## 10. Vérification

Comparer un document silver produit à ce que prédit la spec ci-dessus, sur une entreprise connue.

In [ ]:
{
  "_id": "0201.105.843",
  "enterpriseNumber": "0201.105.843",
  "startDate": "02-03-1956",
  "denominations": {
    "Abréviation": {
      "language": "français",
      "denomination": "I.D.E.A."
    },
    "Dénomination": {
      "language": "français",
      "denomination": "\"I.D.E.A. S.C\""
    }
  },
  "addresses": {
    "Siège": {
      "country": "Belgique",
      "zipcode": "7000",
      "municipality": "Mons",
      "street": "Rue de Nimy",
      "houseNumber": "53",
      "box": ""
    }
  },
  "contacts": {
    "email": "officiel.ic-idea@idea.be"
  },
  "activities": {
    "main": [
      {
        "activityGroup": "Activités ONSS",
        "description": "Administration de et contribution à l’amélioration de l’efficacité des activités économiques",
        "naceVersion": "2025"
      },
      {
        "activityGroup": "Activités ONSS",
        "description": "Administration publique (tutelle) des activités économiques",
        "naceVersion": "2008"
      },
      {
        "activityGroup": "Activités TVA",
        "description": "Études de marché et sondages",
        "naceVersion": "2025"
      },
      {
        "activityGroup": "Activités TVA",
        "description": "Études de marché et sondages d'opinion",
        "naceVersion": "2008"
      },
      {
        "activityGroup": "Activités TVA",
        "description": "Bureau d'étude de marché",
        "naceVersion": "2003"
      }
    ],
    "secondary": [
      {
        "activityGroup": "Activités TVA",
        "description": "Travaux de dragage",
        "naceVersion": "2025"
      }
    ]
  },
  "establishments": {
    "2.382.100.462": {
      "startDate": "01-01-2026",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "Distribution d'eau"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7011",
          "municipality": "Mons",
          "street": "Rue de Baudour (G.)",
          "houseNumber": "SN",
          "box": ""
        }
      },
      "contacts": {},
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Captage, traitement et distribution d’eau",
            "naceVersion": "2025"
          }
        ],
        "secondary": []
      }
    },
    "2.300.665.301": {
      "startDate": "01-04-2020",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "Station d'épuration de Braine-Le-Comte"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7090",
          "municipality": "Braine-le-Comte",
          "street": "Route de Petit Roeulx",
          "houseNumber": "SN",
          "box": ""
        }
      },
      "contacts": {
        "email": "ludovic.delhaye@idea.be",
        "phone": "065/37 57 27"
      },
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Collecte et traitement des eaux usées",
            "naceVersion": "2025"
          }
        ],
        "secondary": []
      }
    },
    "2.343.255.427": {
      "startDate": "01-01-2023",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "Station d’épuration Ecaussinnes"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7190",
          "municipality": "Ecaussinnes",
          "street": "Rue de l'Avedelle",
          "houseNumber": "sn",
          "box": ""
        }
      },
      "contacts": {},
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Collecte et traitement des eaux usées",
            "naceVersion": "2025"
          }
        ],
        "secondary": []
      }
    },
    "2.162.854.629": {
      "startDate": "02-03-1956",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "La Maison de l'Entreprise - Mons"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7000",
          "municipality": "Mons",
          "street": "Rue René Descartes",
          "houseNumber": "2",
          "box": ""
        }
      },
      "contacts": {},
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Autres activités de service de soutien aux entreprises nca",
            "naceVersion": "2025"
          },
          {
            "activityGroup": "Activités ONSSAPL",
            "description": "Intercommunales à vocation générale",
            "naceVersion": "2003"
          },
          {
            "activityGroup": "Activités ONSS",
            "description": "Autres activités de soutien aux entreprises n.c.a.",
            "naceVersion": "2008"
          }
        ],
        "secondary": []
      }
    },
    "2.300.510.990": {
      "startDate": "01-04-2020",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "Station d'épuration de Chapelle-Lez-Herlaimont"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7160",
          "municipality": "Chapelle-lez-Herlaimont",
          "street": "Rue du Vent de Bise",
          "houseNumber": "SN",
          "box": ""
        }
      },
      "contacts": {
        "email": "ludovic.delhaye@idea.be",
        "phone": "065/37 57 27"
      },
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Collecte et traitement des eaux usées",
            "naceVersion": "2025"
          }
        ],
        "secondary": []
      }
    },
    "2.382.106.697": {
      "startDate": "01-01-2026",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "La Maison de l'Entreprise"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7100",
          "municipality": "La Louvière",
          "street": "Rue Arthur Delaby(L.L)",
          "houseNumber": "5",
          "box": ""
        }
      },
      "contacts": {},
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Autres activités de service de soutien aux entreprises nca",
            "naceVersion": "2025"
          }
        ],
        "secondary": []
      }
    },
    "2.165.787.195": {
      "startDate": "02-03-1956",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "Garocentre"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7110",
          "municipality": "La Louvière",
          "street": "Rue Athéna(H-G)",
          "houseNumber": "-",
          "box": ""
        }
      },
      "contacts": {},
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Administration de et contribution à l’amélioration de l’efficacité des activités économiques",
            "naceVersion": "2025"
          },
          {
            "activityGroup": "Activités ONSSAPL",
            "description": "Intercommunales à vocation générale",
            "naceVersion": "2003"
          },
          {
            "activityGroup": "Activités ONSS",
            "description": "Administration publique (tutelle) des activités économiques",
            "naceVersion": "2008"
          }
        ],
        "secondary": []
      }
    },
    "2.162.853.936": {
      "startDate": "02-03-1956",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "I.D.E.A. Mons-Borinage-Centre"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7100",
          "municipality": "La Louvière",
          "street": "Rue Hamoir(L.L)",
          "houseNumber": "30",
          "box": ""
        }
      },
      "contacts": {},
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSSAPL",
            "description": "Autres activités de télécommunication, y compris télédistribution",
            "naceVersion": "2003"
          },
          {
            "activityGroup": "Activités ONSS",
            "description": "Activités de télécommunications filaires, sans fil et satellitaires",
            "naceVersion": "2025"
          },
          {
            "activityGroup": "Activités ONSS",
            "description": "Télécommunications sans fil",
            "naceVersion": "2008"
          }
        ],
        "secondary": []
      }
    },
    "2.300.628.083": {
      "startDate": "01-04-2020",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "Station d'épuaration de Boussoit"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7110",
          "municipality": "La Louvière",
          "street": "Rue de Thieu(BO)",
          "houseNumber": "SN",
          "box": ""
        }
      },
      "contacts": {
        "phone": "065/37 57 27",
        "email": "ludovic.delhaye@idea.be"
      },
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Collecte et traitement des eaux usées",
            "naceVersion": "2025"
          }
        ],
        "secondary": []
      }
    },
    "2.300.626.994": {
      "startDate": "01-04-2020",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "Station d'épuration de Quièvrain"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7380",
          "municipality": "Quiévrain",
          "street": "Rue du Bruil",
          "houseNumber": "SN",
          "box": ""
        }
      },
      "contacts": {
        "email": "ludovic.delhaye@idea.be",
        "phone": "065/37 57 27"
      },
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Collecte et traitement des eaux usées",
            "naceVersion": "2025"
          }
        ],
        "secondary": []
      }
    },
    "2.300.627.489": {
      "startDate": "01-04-2020",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "Station d'épuration de Saint-Vaast"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7100",
          "municipality": "La Louvière",
          "street": "Rue du Moulin à Eau(S-V)",
          "houseNumber": "SN",
          "box": ""
        }
      },
      "contacts": {
        "email": "ludovic.delhaye@idea.be",
        "phone": "065/37 57 27"
      },
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Collecte et traitement des eaux usées",
            "naceVersion": "2025"
          }
        ],
        "secondary": []
      }
    },
    "2.300.492.679": {
      "startDate": "01-04-2020",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "IDEA - UMH"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7000",
          "municipality": "Mons",
          "street": "Boulevard Initialis",
          "houseNumber": "30",
          "box": ""
        }
      },
      "contacts": {
        "email": "ludovic.delhaye@idea.be",
        "phone": "065/37 57 27"
      },
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Administration de et contribution à l’amélioration de l’efficacité des activités économiques",
            "naceVersion": "2025"
          },
          {
            "activityGroup": "Activités ONSS",
            "description": "Administration publique (tutelle) des activités économiques",
            "naceVersion": "2008"
          }
        ],
        "secondary": []
      }
    },
    "2.162.853.540": {
      "startDate": "02-03-1956",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "I.D.E.A. Région Mons-Borinage-Centre"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7180",
          "municipality": "Seneffe",
          "street": "Rue de Soudromont",
          "houseNumber": "1",
          "box": ""
        }
      },
      "contacts": {},
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Collecte et traitement des eaux usées",
            "naceVersion": "2025"
          },
          {
            "activityGroup": "Activités ONSSAPL",
            "description": "Collecte et traitement des eaux usées",
            "naceVersion": "2003"
          }
        ],
        "secondary": []
      }
    },
    "2.300.627.885": {
      "startDate": "01-04-2020",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "Station d'épuration de Trivières"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7100",
          "municipality": "La Louvière",
          "street": "Rue du Provia(TRI)",
          "houseNumber": "SN",
          "box": ""
        }
      },
      "contacts": {
        "email": "ludovic.delhaye@idea.be",
        "phone": "065/37 57 27"
      },
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Collecte et traitement des eaux usées",
            "naceVersion": "2025"
          }
        ],
        "secondary": []
      }
    },
    "2.343.255.130": {
      "startDate": "01-01-2023",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "Station d’épuration Dour"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7370",
          "municipality": "Dour",
          "street": "Rue de Baisieux",
          "houseNumber": "sn",
          "box": ""
        }
      },
      "contacts": {},
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Collecte et traitement des eaux usées",
            "naceVersion": "2025"
          }
        ],
        "secondary": []
      }
    },
    "2.343.255.823": {
      "startDate": "01-01-2023",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "Station d’épuration Morlanwelz"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7140",
          "municipality": "Morlanwelz",
          "street": "Rue des Haignies(MLZ)",
          "houseNumber": "sn",
          "box": ""
        }
      },
      "contacts": {},
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Collecte et traitement des eaux usées",
            "naceVersion": "2025"
          }
        ],
        "secondary": []
      }
    },
    "2.162.853.342": {
      "startDate": "02-03-1956",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "I.D.E.A. Région Mons-Borinage-Centre"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7390",
          "municipality": "Quaregnon",
          "street": "Chasse des Prés",
          "houseNumber": "1",
          "box": ""
        }
      },
      "contacts": {},
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Collecte et traitement des eaux usées",
            "naceVersion": "2025"
          },
          {
            "activityGroup": "Activités ONSSAPL",
            "description": "Collecte et traitement des eaux usées",
            "naceVersion": "2003"
          }
        ],
        "secondary": []
      }
    },
    "2.162.852.253": {
      "startDate": "02-03-1956",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "I.D.E.A. Région Mons-Borinage-Centre"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7000",
          "municipality": "Mons",
          "street": "Rue de Nimy",
          "houseNumber": "53",
          "box": ""
        }
      },
      "contacts": {},
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Administration publique (tutelle) des activités économiques",
            "naceVersion": "2008"
          },
          {
            "activityGroup": "Activités ONSSAPL",
            "description": "Intercommunales à vocation générale",
            "naceVersion": "2003"
          },
          {
            "activityGroup": "Activités ONSS",
            "description": "Administration de et contribution à l’amélioration de l’efficacité des activités économiques",
            "naceVersion": "2025"
          }
        ],
        "secondary": []
      }
    },
    "2.343.255.724": {
      "startDate": "01-01-2023",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "Station d’épuration Godarville"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7160",
          "municipality": "Chapelle-lez-Herlaimont",
          "street": "Rue du Castia",
          "houseNumber": "sn",
          "box": ""
        }
      },
      "contacts": {},
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Collecte et traitement des eaux usées",
            "naceVersion": "2025"
          }
        ],
        "secondary": []
      }
    },
    "2.162.854.926": {
      "startDate": "02-03-1956",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "La Maison de l'Entreprise - Binche"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7130",
          "municipality": "Binche",
          "street": "Rue des Pastures(BIN)",
          "houseNumber": "95",
          "box": ""
        }
      },
      "contacts": {},
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Autres activités de service de soutien aux entreprises nca",
            "naceVersion": "2025"
          },
          {
            "activityGroup": "Activités ONSS",
            "description": "Autres activités de soutien aux entreprises n.c.a.",
            "naceVersion": "2008"
          },
          {
            "activityGroup": "Activités ONSSAPL",
            "description": "Intercommunales à vocation générale",
            "naceVersion": "2003"
          }
        ],
        "secondary": []
      }
    },
    "2.343.255.328": {
      "startDate": "01-01-2023",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "Station d’épuration Frameries"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7080",
          "municipality": "Frameries",
          "street": "Rue des Fours-à-Chaux",
          "houseNumber": "sn",
          "box": ""
        }
      },
      "contacts": {},
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Collecte et traitement des eaux usées",
            "naceVersion": "2025"
          }
        ],
        "secondary": []
      }
    },
    "2.300.627.291": {
      "startDate": "01-04-2020",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "Station d'épuration de Soignies Biamont"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7060",
          "municipality": "Soignies",
          "street": "Chemin de la Platinerie",
          "houseNumber": "SN",
          "box": ""
        }
      },
      "contacts": {
        "email": "ludovic.delhaye@idea.be",
        "phone": "065/37 57 27"
      },
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Collecte et traitement des eaux usées",
            "naceVersion": "2025"
          }
        ],
        "secondary": []
      }
    },
    "2.162.853.639": {
      "startDate": "02-03-1956",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "I.D.E.A. Région Mons-Borinage-Centre"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7110",
          "municipality": "La Louvière",
          "street": "Rue Grand Peuplier(H-A)",
          "houseNumber": "20",
          "box": ""
        }
      },
      "contacts": {},
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Télécommunications sans fil",
            "naceVersion": "2008"
          },
          {
            "activityGroup": "Activités ONSS",
            "description": "Activités de télécommunications filaires, sans fil et satellitaires",
            "naceVersion": "2025"
          },
          {
            "activityGroup": "Activités ONSSAPL",
            "description": "Autres activités de télécommunication, y compris télédistribution",
            "naceVersion": "2003"
          }
        ],
        "secondary": []
      }
    },
    "2.227.021.515": {
      "startDate": "01-01-2014",
      "denominations": {
        "Dénomination commerciale": {
          "language": "français",
          "denomination": "Service entretien des biens - Infrastructures"
        }
      },
      "addresses": {
        "Unité d'établissement": {
          "country": "Belgique",
          "zipcode": "7033",
          "municipality": "Mons",
          "street": "Rue de Ciply (C.)",
          "houseNumber": "265",
          "box": "B"
        }
      },
      "contacts": {},
      "activities": {
        "main": [
          {
            "activityGroup": "Activités ONSS",
            "description": "Administration de et contribution à l’amélioration de l’efficacité des activités économiques",
            "naceVersion": "2025"
          },
          {
            "activityGroup": "Activités ONSS",
            "description": "Administration publique (tutelle) des activités économiques",
            "naceVersion": "2008"
          }
        ],
        "secondary": []
      }
    }
  },
  "branches": {},
  "juridicalForm": "Société coopérative",
  "juridicalSituation": "Situation normale",
  "status": "Actif",
  "typeOfEnterprise": "Personne morale"
}

{'_id': '0201.105.843',
 'enterpriseNumber': '0201.105.843',
 'startDate': '02-03-1956',
 'denominations': {'Abréviation': {'language': 'français',
   'denomination': 'I.D.E.A.'},
  'Dénomination': {'language': 'français', 'denomination': '"I.D.E.A. S.C"'}},
 'addresses': {'Siège': {'country': 'Belgique',
   'zipcode': '7000',
   'municipality': 'Mons',
   'street': 'Rue de Nimy',
   'houseNumber': '53',
   'box': ''}},
 'contacts': {'email': 'officiel.ic-idea@idea.be'},
 'activities': {'main': [{'activityGroup': 'Activités ONSS',
    'description': 'Administration de et contribution à l’amélioration de l’efficacité des activités économiques',
    'naceVersion': '2025'},
   {'activityGroup': 'Activités ONSS',
    'description': 'Administration publique (tutelle) des activités économiques',
    'naceVersion': '2008'},
   {'activityGroup': 'Activités TVA',
    'description': 'Études de marché et sondages',
    'naceVersion': '2025'},
   {'activityGroup': 'Activités TVA',
    'description

In [ ]:
import pprint
from pymongo import MongoClient

# Connexion à la base de données
db = MongoClient().kbo_database

# Récupération du document transformé (couche Silver) pour l'entreprise cible
doc_silver = db.entreprise_silver.find_one({"_id": "0201.105.843"})

if doc_silver:
    print("Document Silver généré : \n")
    # pprint permet un affichage indenté et lisible du dictionnaire
    pprint.pprint(doc_silver, sort_dicts=False)
else:
    print("Le document est introuvable. Vérifie que l'étape 9 s'est bien exécutée !")

Le document est introuvable. Vérifie que l'étape 9 s'est bien exécutée !


## 12. Schéma cible de `entreprise_silver`

| Champ | Type | Origine / règle |
|---|---|---|
| `_id` | string | copié tel quel du bronze |
| `enterpriseNumber` | string | `EnterpriseNumber` (ou `_id` en secours) |
| `startDate` | string, optionnel | copié si présent |
| `status`, `juridicalSituation`, `typeOfEnterprise`, `juridicalForm`, `juridicalFormCAC` | string, optionnels | traduits FR via `code.csv`, omis si absents du bronze |
| `denominations` | dict `{type traduit: {language, denomination}}` | dernier gagne en cas de type dupliqué |
| `addresses` | dict `{type traduit: {country, zipcode, municipality, street, houseNumber, box}}` | champs vides omis, `country` nettoyé + défaut `"Belgique"` |
| `contacts` | dict `{email?, phone?, web?}` | `EntityContact` jamais lu |
| `activities` | `{main: [...], secondary: [...]}` | dédoublonné par `(activityGroup, description)`, version NACE la plus récente gagne, `naceCode` brut jamais gardé |
| `establishments` | dict `{EstablishmentNumber: {startDate?, denominations, addresses, contacts, activities}}` | mêmes règles que l'entreprise, sans `EnterpriseNumber` |
| `branches` | dict `{Id: {startDate?, addresses, contacts}}` | sans `denominations` ni `activities` (une succursale n'en a jamais) |

In [10]:
import csv
import re
from pymongo import MongoClient

# ==========================================
# 0. CHARGEMENT DU RÉFÉRENTIEL DE TRADUCTION
# ==========================================

translations = {}
path_code = "data/code.csv"

with open(path_code, mode='r', encoding='utf-8') as file:
    reader = csv.DictReader(file)
    for row in reader:
        if row.get("Language") == "FR":
            category = row.get("Category")
            code = row.get("Code")
            description = row.get("Description")
            
            if category not in translations:
                translations[category] = {}
            translations[category][code] = description

def translate(category_name, code_value):
    if not code_value:
        return ""
    return translations.get(category_name, {}).get(code_value, code_value)

# ==========================================
# 1. NETTOYAGE ET TRANSFORMATIONS
# ==========================================

def process_scalars(bronze_doc):
    silver_doc = {}
    silver_doc["_id"] = bronze_doc.get("_id")
    silver_doc["enterpriseNumber"] = bronze_doc.get("EnterpriseNumber", silver_doc["_id"])
    
    if "StartDate" in bronze_doc:
        silver_doc["startDate"] = bronze_doc.get("StartDate")
        
    scalar_fields_to_map = [
        ("Status", "status"),
        ("JuridicalSituation", "juridicalSituation"),
        ("TypeOfEnterprise", "typeOfEnterprise"),
        ("JuridicalForm", "juridicalForm"),
        ("JuridicalFormCAC", "juridicalFormCAC")
    ]
    
    for bronze_key, silver_key in scalar_fields_to_map:
        val = bronze_doc.get(bronze_key)
        if val: 
            silver_doc[silver_key] = translate(bronze_key, val)
    return silver_doc

def process_denominations(denominations_bronze):
    denoms_silver = {}
    for d in denominations_bronze:
        t_type = translate("TypeOfDenomination", d.get("TypeOfDenomination"))
        denoms_silver[t_type] = {
            "language": d.get("Language"),
            "denomination": d.get("Denomination")
        }
    return denoms_silver

def clean_country(country_fr):
    if not country_fr:
        return "Belgique"
    cleaned = re.sub(r'\(.*?\)', '', country_fr)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned if cleaned else "Belgique"

def process_addresses(addresses_bronze):
    addrs_silver = {}
    for a in addresses_bronze:
        t_type = translate("TypeOfAddress", a.get("TypeOfAddress"))
        addr = {}
        country = clean_country(a.get("CountryFR"))
        if country: addr["country"] = country
        
        for field in ["Zipcode", "Municipality", "Street", "HouseNumber", "Box"]:
            val = a.get(field)
            if val:
                addr[field[0].lower() + field[1:]] = val
        addrs_silver[t_type] = addr
    return addrs_silver

def process_contacts(contacts_bronze):
    contacts_silver = {}
    for c in contacts_bronze:
        c_type = c.get("ContactType", "").lower()
        if c_type == "entitycontact": 
            continue 
        val = c.get("Value")
        if val: 
            contacts_silver[c_type] = val
    return contacts_silver

def process_activities(activities_bronze):
    dedup_dict = {}
    for act in activities_bronze:
        group = act.get("ActivityGroup")
        desc = act.get("Description")
        version = act.get("NaceVersion", "0")
        classification = act.get("Classification", "main").lower()
        
        key = (group, desc)
        
        if key not in dedup_dict or int(version) > int(dedup_dict[key]["naceVersion"]):
            dedup_dict[key] = {
                "activityGroup": group,
                "description": desc,
                "naceVersion": version,
                "_classification": classification
            }
            
    activities_silver = {"main": [], "secondary": []}
    for act in dedup_dict.values():
        cat = "main" if act.pop("_classification") == "main" else "secondary"
        activities_silver[cat].append(act)
        
    return activities_silver

def process_establishments(establishments_bronze):
    ests_silver = {}
    for e in establishments_bronze:
        est_id = e.get("EstablishmentNumber")
        est_data = {}
        if "StartDate" in e:
            est_data["startDate"] = e["StartDate"]
            
        est_data["denominations"] = process_denominations(e.get("Denominations", []))
        est_data["addresses"] = process_addresses(e.get("Addresses", []))
        est_data["contacts"] = process_contacts(e.get("Contacts", []))
        est_data["activities"] = process_activities(e.get("Activities", []))
        
        ests_silver[est_id] = est_data
    return ests_silver

def process_branches(branches_bronze):
    branches_silver = {}
    for b in branches_bronze:
        branch_id = b.get("Id")
        branch_data = {}
        if "StartDate" in b:
            branch_data["startDate"] = b["StartDate"]
            
        branch_data["addresses"] = process_addresses(b.get("Addresses", []))
        branch_data["contacts"] = process_contacts(b.get("Contacts", []))
        
        branches_silver[branch_id] = branch_data
    return branches_silver

def transform_to_silver(bronze_doc):
    silver_doc = process_scalars(bronze_doc)
    silver_doc["denominations"] = process_denominations(bronze_doc.get("Denominations", []))
    silver_doc["addresses"] = process_addresses(bronze_doc.get("Addresses", []))
    silver_doc["contacts"] = process_contacts(bronze_doc.get("Contacts", []))
    silver_doc["activities"] = process_activities(bronze_doc.get("Activities", []))
    silver_doc["establishments"] = process_establishments(bronze_doc.get("Establishments", []))
    silver_doc["branches"] = process_branches(bronze_doc.get("Branches", []))
    return silver_doc

# ==========================================
# 2. EXÉCUTION DU PIPELINE VERS MONGODB (Limité à 100)
# ==========================================

client = MongoClient("mongodb://localhost:27017/")
db = client["kbo_db"]

# Vider la collection silver pour repartir au propre
db.entreprise_silver.delete_many({}) 

# On lance la transformation sur le Bronze (100 docs max)
for doc in db.entreprise.find().limit(10000):
    silver_doc = transform_to_silver(doc)
    db.entreprise_silver.insert_one(silver_doc)

print(f"✅ Terminé ! {db.entreprise_silver.count_documents({})} documents insérés dans le silver.")

✅ Terminé ! 10000 documents insérés dans le silver.


In [11]:
import pprint
from pymongo import MongoClient

# Connexion à MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["kbo_db"]

# On va chercher le tout premier document inséré dans entreprise_silver
doc_sample = db.entreprise_silver.find_one()

print("🔍 Exemple de document Silver généré depuis MongoDB :\n")
pprint.pprint(doc_sample, sort_dicts=False)

🔍 Exemple de document Silver généré depuis MongoDB :

{'_id': ObjectId('6a676bc449d1f33a2397b64d'),
 'enterpriseNumber': '0200.065.765',
 'startDate': '09-08-1960',
 'status': 'Actif',
 'juridicalSituation': 'Situation normale',
 'typeOfEnterprise': 'Personne morale',
 'juridicalForm': 'Association prestataire de services (Région flamande)',
 'denominations': {},
 'addresses': {},
 'contacts': {},
 'activities': {'main': [], 'secondary': []},
 'establishments': {},
 'branches': {}}
